In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import numpy as np

In [2]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [3]:
dataset = datasets.ImageFolder(
    root="./data/images_128",
    transform=transform
)

In [4]:
dataloader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True,
    persistent_workers=True,
    prefetch_factor=2,
    num_workers=2,
    pin_memory=True
)

In [5]:
import time


for i, (x, y) in enumerate(dataloader):
    start = time.time()
    print(f"{i+1} batch load time: {time.time()-start}")
    if i==1: break

1 batch load time: 2.384185791015625e-06
2 batch load time: 2.1457672119140625e-06


In [6]:
x, y = next(iter(dataloader))
print(x.mean(), x.std())

tensor(0.4477) tensor(0.2589)


In [7]:
len(dataloader)

719

In [8]:
class CnnModel (nn.Module):
    def __init__(self):
        # initialize super 
        super().__init__()
        
        # Kernel
        self.kernel1 = nn.Conv2d(
            in_channels=3,
            out_channels=8,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.bn1 = nn.BatchNorm2d(8)
        
        self.kernel2 = nn.Conv2d(
            in_channels=8,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.bn2 = nn.BatchNorm2d(16)
        
        self.kernel3 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.bn3 = nn.BatchNorm2d(32)
        
        self.kernel4 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.bn4 = nn.BatchNorm2d(64)
        
        # Linear neural networks
        self.W1 = nn.Parameter(torch.randn(64*8*8, 2) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(2))

        # loss function
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def forward(self, x):
        
        # first kernel
        x = self.kernel1(x)
        x = self.bn1(x)
        x = nn.functional.relu(x)
        x = nn.functional.max_pool2d(x, 2)
        
        # second kernel
        x = self.kernel2(x)
        x = self.bn2(x)
        x = nn.functional.relu(x)
        x = nn.functional.max_pool2d(x, 2)
        
        # third kernel
        x = self.kernel3(x)
        x = self.bn3(x)
        x = nn.functional.relu(x)
        x = nn.functional.max_pool2d(x, 2)
        
        # fourth kernel
        x = self.kernel4(x)
        x = self.bn4(x)
        x = nn.functional.relu(x)
        x = nn.functional.max_pool2d(x, 2)
                
        # reshape images
        x = x.reshape((x.shape[0], -1))
        
        x = x @ self.W1 + self.b1
                
        return x
    
    def compute_loss(self, y_hat, y):        
        # loss calling
        loss = self.criterion(y_hat, y)
        return loss
    
    def backward_propagation(self, loss, learning_rate=0.01):
        # start loss backward
        loss.backward()
        
        with torch.no_grad():
            for param in self.parameters():
                if param.grad is not None:
                    param.data -= learning_rate * param.grad
                    param.grad.zero_()
        
        return loss.item()

In [9]:
def train_epoch(model, dataloader,learning_rate=0.001):
    # using for loop to access each batch
    costs = []
    for x, y in dataloader:
        # transfer to cuda
        x = x.to("cuda", non_blocking=True)
        y = y.to("cuda", non_blocking=True)
        
        #code to stop tracking grad during backtrack
        y_hat = model.forward(x)
        loss = model.compute_loss(y_hat, y)
        cost = model.backward_propagation(loss, learning_rate)
        costs.append(cost)
    
    return costs

In [10]:
# initialize model
model = CnnModel().to("cuda")

In [11]:
import time
import numpy as np

costs = []

In [ ]:
for i in range(0, 100):
    #start time
    start = time.time()
    epoch = train_epoch(model, dataloader=dataloader, learning_rate=0.005)
    #end time
    end = time.time()
    cost = np.mean(np.array(epoch))
    
    costs.append(cost)
    
    if (i+1)%10 ==0:
        # model save name
        file_name = f"model_epoch_{i+1}"
        # print loss in every 10 epochs        
        print(f"{(i+1)} Epoch Loss ====> {cost} in {end-start}s per epoch.")
        # save the model
        torch.save(model.state_dict(), f"./model/{file_name}.pth")
        print(f"Saved model {file_name}")
        

10 Epoch Loss ====> 0.4609673984027539 in 7.356410503387451s per epoch.
Saved model model_epoch_10
20 Epoch Loss ====> 0.37799641788420324 in 6.635260343551636s per epoch.
Saved model model_epoch_20


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

epoch = [i+1 for i in range(len(costs))]

display(len(costs))
display(len(epoch))

In [ ]:

df = pd.DataFrame({"x":epoch, "y":costs})

plt.figure(figsize=(10,5))
plt.title("Loss Drop function!")
plt.ylabel("Loss")
plt.xlabel("Epoch")
sns.lineplot(df, x="x", y="y")

In [ ]:
#test model

test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

test_dataset = datasets.ImageFolder(
    root="./data/valid",
    transform=test_transforms
)

test_dataloader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=2,
    pin_memory=True, 
    prefetch_factor=2, 
    persistent_workers=True
)

print(len(test_dataset))
print(len(test_dataloader))

In [ ]:
def evaluate_model(model, dataloader, device="cuda"):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0

    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            # loss (optional but useful)
            loss = criterion(outputs, y)
            total_loss += loss.item()

            # predictions
            preds = torch.argmax(outputs, dim=1)

            # store results
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())

    # combine all batches
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    avg_loss = total_loss / len(dataloader)

    return all_preds, all_labels, avg_loss

In [ ]:
def classification_metrics(y_pred, y_true):
    """
    y_pred: tensor of predicted labels (0/1)
    y_true: tensor of true labels (0/1)
    """

    y_pred = y_pred.cpu()
    y_true = y_true.cpu()

    tp = ((y_pred == 1) & (y_true == 1)).sum().item()
    tn = ((y_pred == 0) & (y_true == 0)).sum().item()
    fp = ((y_pred == 1) & (y_true == 0)).sum().item()
    fn = ((y_pred == 0) & (y_true == 1)).sum().item()

    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

    print("==== Confusion Matrix ====")
    print(f"TP: {tp}  FP: {fp}")
    print(f"FN: {fn}  TN: {tn}")
    print()
    print("==== Metrics ====")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    return {
    "tp": tp,
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1
    }

In [ ]:
scores = []

for i in range(10):
    #load model
    test_model = CnnModel().to("cuda")
    test_model.load_state_dict(torch.load(f"./model/model_epoch_{(i+1)*10}.pth"))
    
    y_pred, y_true, avg_loss = evaluate_model(test_model, test_dataloader)
    print(f"model_epoch_{(i+1)*10} Avg Loss => {avg_loss}")
    
    score = classification_metrics(y_pred=y_pred, y_true=y_true)
    scores.append(score)
    
    print(f"\n\nmodel_epoch_{(i+1)*10} scores:")
    print(score)